# Build shuffled dataset for LLM labeling

Combine Telegram JSONL exports into one JSON array. Each record contains only `text` and `target`; source, date, URLs, views, and Telegram IDs are omitted. `target` is a dictionary keyed by the 30 business category names plus the `Реклама/мусор` quality label, initialized to zero. The notebook preserves duplicate posts and skips only blank text. A fixed seed makes the shuffle reproducible.

In [ ]:
from pathlib import Path
import json
import random

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = ROOT / "data" / "raw"
OUTPUT_PATH = ROOT / "data" / "labeling_dataset.json"
CATEGORY_DOC = next(ROOT.glob("Категории*новост*.md"))
SHUFFLE_SEED = 20261021

categories = {}
for line in CATEGORY_DOC.read_text(encoding="utf-8").splitlines():
    cells = [cell.strip() for cell in line.split("|")]
    if len(cells) >= 4 and cells[1].isdigit():
        categories[int(cells[1])] = cells[2]

expected_ids = set(range(1, 32))
if set(categories) != expected_ids or len(set(categories.values())) != 31:
    raise ValueError(f"Expected exactly 31 uniquely named labels in {CATEGORY_DOC}")
category_names = [categories[index] for index in range(1, 32)]

files = sorted(INPUT_DIR.glob("*.jsonl"))
if not files:
    raise FileNotFoundError(f"No JSONL files found in {INPUT_DIR}")

print(f"Input files: {len(files)}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
records = []
blank_text_count = 0
malformed_count = 0
malformed_examples = []

for path in files:
    with path.open("r", encoding="utf-8") as source:
        for line_number, line in enumerate(source, start=1):
            if not line.strip():
                continue
            try:
                post = json.loads(line)
            except json.JSONDecodeError as exc:
                malformed_count += 1
                if len(malformed_examples) < 10:
                    malformed_examples.append((str(path), line_number, str(exc)))
                continue

            text = post.get("text")
            if not isinstance(text, str) or not text.strip():
                blank_text_count += 1
                continue

            records.append({
                "text": text,
                "target": {name: 0 for name in category_names},
            })

if malformed_count:
    raise ValueError(
        f"Found {malformed_count} malformed JSONL rows. Examples: {malformed_examples}"
    )

random.Random(SHUFFLE_SEED).shuffle(records)

print(f"Rows ready: {len(records):,}")
print(f"Blank rows skipped: {blank_text_count:,}")
print(f"Malformed rows: {malformed_count}")

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as output:
    json.dump(records, output, ensure_ascii=False, separators=(",", ":"))

print(f"Wrote {len(records):,} rows ({OUTPUT_PATH.stat().st_size:,} bytes)")

In [ ]:
# Validate the exported structure and confirm that no source metadata remains.
with OUTPUT_PATH.open("r", encoding="utf-8") as source:
    check = json.load(source)

assert isinstance(check, list)
assert len(check) == len(records)
assert all(set(row) == {"text", "target"} for row in check)
assert all(isinstance(row["text"], str) for row in check)
assert all(list(row["target"]) == category_names for row in check)
assert all(value == 0 for row in check for value in row["target"].values())

print("Validation passed")
first_record = check[0]
print("Example record:", {"text": first_record["text"][:180], "target": first_record["target"]})